In [ ]:
!pip install torch==2.6.0 --quiet
!pip install numpy==2.2.3 --quiet
!pip install pandas==2.2.3 --quiet
!pip install matplotlib==3.10.1 --quiet
!pip install scikit-learn==1.6.0 --quiet
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.linear_model import Lasso
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

RFF

In [ ]:
GLOBAL_SEED = 7 #изменять данный seed
torch.manual_seed(GLOBAL_SEED)
torch.set_num_threads(4)

ds = [2, 5, 10, 20]
ms = [32, 64, 128, 256, 512, 1024]
n_train_base = 50000
n_test = 10000

def rff_features(X, omega, b):
    return np.sqrt(2.0 / omega.shape[0]) * torch.cos(X @ omega.T + b)

def ridge_regression(Z, y, lam):
    n, m = Z.shape
    I = torch.eye(m, device=Z.device, dtype=Z.dtype)
    return torch.linalg.solve(Z.T @ Z + n * lam * I, Z.T @ y)

def f_radial(X):
    d = X.shape[1]
    c = 0.5 * torch.ones(d)
    return torch.exp(-0.5 * torch.sum((X - c)**2, dim=1))

def f_ridge_sum(X):
    d = X.shape[1]
    a1 = torch.tensor([1, 1] + [0.5]*(d-2), dtype=torch.float32)[:d]
    a2 = torch.tensor([0.5, 1, 1] + [0.5]*(d-3), dtype=torch.float32)[:d]
    a3 = torch.tensor([1, 0.5, 1, 1] + [1]*(d-4), dtype=torch.float32)[:d]
    a1 = a1 / torch.norm(a1)
    a2 = a2 / torch.norm(a2)
    a3 = a3 / torch.norm(a3)
    b1,b2,b3 = 0.1,0.7,0.3
    return torch.tanh(X @ a1 + b1) + torch.sin(X @ a2 + b2) + torch.cos(X @ a3 + b3)

def f_fourier_narrow(X):
    d = X.shape[1]
    w1 = torch.ones(d)
    w2 = torch.tensor([2,1.5] + [1]*(d-2))[:d]
    return torch.cos(X @ w1) + 0.5*torch.sin(X @ w1) + 0.8*torch.cos(X @ w2) - 0.3*torch.sin(X @ w2)

def f_fourier_noisy(X):
    d = X.shape[1]
    w1 = torch.ones(d)
    w2 = torch.ones(d)*10
    return torch.cos(X @ w1) + 0.5*torch.sin(X @ w1) + 0.8*torch.cos(X @ w2) - 0.3*torch.sin(X @ w2)

names = ["Radial", "Ridge sum", "Fourier narrow", "Fourier noisy"]
funcs = [f_radial, f_ridge_sum, f_fourier_narrow, f_fourier_noisy]

results = []

for d in ds:
    gen_train = torch.Generator().manual_seed(GLOBAL_SEED + 1000*d)
    gen_test  = torch.Generator().manual_seed(GLOBAL_SEED + 2000*d)
    n_train = n_train_base * d
    x_train = torch.rand(n_train, d, generator=gen_train)
    x_test  = torch.rand(n_test, d, generator=gen_test)

    for f, name in zip(funcs, names):
        y_train = f(x_train)
        y_test  = f(x_test)

        for m in ms:
            ridge_lambda = 1 / (m)
            gen_rff = torch.Generator().manual_seed(GLOBAL_SEED + 1000*d + m)
            sigma = 0.05 if name == "Fourier noisy" else np.sqrt(d)

            omega = torch.randn(m, d, generator=gen_rff) / sigma
            b = 2 * np.pi * torch.rand(m, generator=gen_rff)

            start_time = time.time()
            phi_train = rff_features(x_train, omega, b)
            phi_test = rff_features(x_test, omega, b)

            theta = ridge_regression(phi_train, y_train, ridge_lambda)
            train_time = time.time() - start_time

            y_pred = phi_test @ theta
            l2_error = torch.sqrt(torch.mean((y_pred - y_test) ** 2)).item()

            results.append({
                "function": name,
                "d": d,
                "m": m,
                "L2_error": l2_error,
                "train_time_sec": train_time
            })

            print(f" {name:14} d={d:2d}, m={m:4d}, L2={l2_error:.8f}, time={train_time:.3f}s")

df = pd.DataFrame(results)

for name in names:
    plt.figure(figsize=(10,6))
    subset = df[df["function"] == name]
    for d in ds:
        sub_d = subset[subset["d"]==d]
        plt.plot(sub_d["m"], sub_d["L2_error"], marker='o', label=f"d={d}")
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel("Число признаков $m$")
    plt.ylabel("Тестовая $L^2$-ошибка")
    plt.title(f"RFF + Ridge: {name}")
    plt.grid(True, which="both", ls="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()
results = []

MLP

In [ ]:
GLOBAL_SEED = 7 #изменять этот seed
torch.manual_seed(GLOBAL_SEED)
torch.set_num_threads(4)

ds = [2, 5, 10, 20]
ms = [32, 64, 128, 256, 512, 1024]
n_train_base = 50000
n_test = 10000
batch_size = 512
weight_decay = 1e-4
max_epochs_base = 10000

lr_dict = {
    "Radial": 1e-3,
    "Ridge sum": 1e-3,
    "Fourier narrow": 1e-3,
    "Fourier noisy": 1e-2
}

def f_radial(x):
    c = 0.5 * torch.ones(x.shape[1])
    return torch.exp(-0.5 * torch.sum((x - c)**2, dim=1))

def f_ridge_sum(x):
    d = x.shape[1]
    a1 = torch.tensor([1,1]+[0.5]*(d-2), dtype=torch.float32)[:d]
    a2 = torch.tensor([0.5,1,1]+[0.5]*(d-3), dtype=torch.float32)[:d]
    a3 = torch.tensor([1,0.5,1,1]+[1]*(d-4), dtype=torch.float32)[:d]
    a1, a2, a3 = a1/a1.norm(), a2/a2.norm(), a3/a3.norm()
    b1, b2, b3 = 0.1, 0.7, 0.3
    return torch.tanh(x@a1+b1) + torch.sin(x@a2+b2) + torch.cos(x@a3+b3)

def f_fourier_narrow(x):
    d = x.shape[1]
    w1 = torch.ones(d)
    w2 = torch.tensor([2,1.5]+[1]*(d-2))
    return torch.cos(x@w1) + 0.5*torch.sin(x@w1) + 0.8*torch.cos(x@w2) - 0.3*torch.sin(x@w2)

def f_fourier_noisy(x):
    d = x.shape[1]
    w1 = torch.ones(d)
    w2 = torch.ones(d)*10
    return torch.cos(x@w1) + 0.5*torch.sin(x@w1) + 0.8*torch.cos(x@w2) - 0.3*torch.sin(x@w2)

funcs = [f_radial, f_ridge_sum, f_fourier_narrow, f_fourier_noisy]
names = ["Radial", "Ridge sum", "Fourier narrow", "Fourier noisy"]

results = []

for d in ds:
    gen_data = torch.Generator().manual_seed(GLOBAL_SEED + 1000*d)
    gen_batch = torch.Generator().manual_seed(GLOBAL_SEED + 1000*d + 1)
    n_train = n_train_base * d

    x_train = (torch.rand(n_train, d, generator=gen_data)-0.5)*2
    x_test  = (torch.rand(n_test, d, generator=gen_data)-0.5)*2

    for f, name in zip(funcs, names):
        y_train = f(x_train)
        y_test  = f(x_test)

        for m in ms:
            model = nn.Sequential(nn.Linear(d, m), nn.Tanh(), nn.Linear(m, 1))

            base_lr = lr_dict[name]
            lr_m = base_lr * (32/m)**0.25

            optimizer = optim.Adam(model.parameters(), lr=lr_m, weight_decay=weight_decay)
            loss_fn = nn.MSELoss()

            num_epochs = max_epochs_base + 5000*(m//256)
            steps_per_epoch = n_train // batch_size
            total_steps = num_epochs * steps_per_epoch

            scheduler = optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=lr_m, total_steps=total_steps, pct_start=0.1,
                anneal_strategy='cos', div_factor=25.0, final_div_factor=100.0
            )

            start_time = time.time()
            for epoch in range(num_epochs):
                idx = torch.randint(0, n_train, (batch_size,), generator=gen_batch)
                xb, yb = x_train[idx], y_train[idx].unsqueeze(1)

                optimizer.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            train_time = time.time() - start_time

            with torch.no_grad():
                y_pred = model(x_test)
                l2_error = torch.sqrt(torch.mean((y_pred - y_test.unsqueeze(1))**2)).item()

            results.append({"function": name, "d": d, "m": m,
                            "L2_error": l2_error, "train_time_sec": train_time})
            print(f"{name:14} d={d:2d}, m={m:4d}, L2={l2_error:.6f}, time={train_time:.1f}s, lr={lr_m:.6f}")

df = pd.DataFrame(results)
functions = df['function'].unique()
colors = ['r', 'g', 'b', 'm', 'c', 'y']

for func in functions:
    plt.figure(figsize=(8,5))
    df_func = df[df['function'] == func]

    for i, d in enumerate(sorted(df_func['d'].unique())):
        df_plot = df_func[df_func['d'] == d].sort_values('m')
        plt.plot(df_plot['m'], df_plot['L2_error'], marker='o', color=colors[i], label=f'd={d}')

    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel("m (ширина сети)")
    plt.ylabel("L2 ошибка")
    plt.title(f"L2 ошибка vs m для функции '{func}'")
    plt.legend()
    plt.grid(True, which="both", ls="--", alpha=0.6)
    plt.show()

Barron norm

In [ ]:
GLOBAL_SEED = 7
def barron_proxy(f, d, N=25000, base_M=60,
                 scales_mult=None, alpha=None, seed=GLOBAL_SEED):
    scales = [0.3, 1.0, 3.0, 5.0, 10.0, 50.0]
    alpha = 1e-6 if d <= 5 else 1e-5

    np.random.seed(seed)
    X = np.random.rand(N, d).astype(np.float32)
    y = f(X).astype(np.float32)
    y = y - np.mean(y)

    all_omegas = []
    all_Z = []

    for s in scales:
        M = int(base_M * d)
        omega = np.random.randn(M, d) * s
        b = np.random.uniform(0, 2 * np.pi, M).astype(np.float32)

        Z = np.sqrt(2.0 / M) * np.cos(X @ omega.T + b)
        all_omegas.append(omega)
        all_Z.append(Z)

    Z_all = np.hstack(all_Z)
    omega_all = np.vstack(all_omegas)

    lasso = Lasso(
        alpha=alpha,
        max_iter=7000,
        tol=1e-4,
        fit_intercept=False,
        random_state=seed
    )
    lasso.fit(Z_all, y)
    a = lasso.coef_

    return np.sum(np.abs(a) * np.linalg.norm(omega_all, axis=1))
def f_radial(X):
    return np.exp(-2 * np.sum((X-0.5)**2, axis=1))

def f_ridge_sum(X):
    d = X.shape[1]
    a1 = np.array([1,1] + [0.5]*(d-2))[:d]
    a2 = np.array([0.5,1,1] + [0.5]*(d-3))[:d]
    a3 = np.array([1,0.5,1,1] + [1]*(d-4))[:d]
    b1,b2,b3 = 0.1,0.7,0.3
    return np.tanh(X @ a1 + b1) + np.sin(X @ a2 + b2) + np.cos(X @ a3 + b3)

def f_fourier_narrow(X):
    d = X.shape[1]
    w1 = np.ones(d)
    w2 = np.array([2,1.5] + [1]*(d-2))[:d]
    return np.cos(X @ w1) + 0.5*np.sin(X @ w1) + 0.8*np.cos(X @ w2) - 0.3*np.sin(X @ w2)

def f_fourier_noisy(X):
    d = X.shape[1]
    w1 = np.ones(d)
    w2 = np.ones(d)*10
    return np.cos(X @ w1) + 0.5*np.sin(X @ w1) + 0.8*np.cos(X @ w2) - 0.3*np.sin(X @ w2)

names = ["Radial", "Ridge sum", "Fourier narrow", "Fourier noisy"]
funcs = [f_radial, f_ridge_sum, f_fourier_narrow, f_fourier_noisy]
ds = [2, 5,10, 20]

print("Proxy-норма Баррона:\n")
for d in ds:
    print(f"d = {d}")
    for f, name in zip(funcs, names):
        a = 1e-5
        proxy = barron_proxy(f, d, N=1000*d, alpha=a)
        print(f"{name:18} : proxy-norm ≈ {proxy:.3f}")

Depth separation

In [ ]:
GLOBAL_SEED=7
torch.manual_seed(GLOBAL_SEED)

n_samples = 1000
X = torch.linspace(-1, 1, n_samples).unsqueeze(1)
y = torch.exp(-torch.abs(X))
for i in [40,200]:
    one_layer = nn.Sequential(
        nn.Linear(1, i),
        nn.Tanh(),
        nn.Linear(i, 1)
    )

    two_layer = nn.Sequential(
        nn.Linear(1, 9),
        nn.Tanh(),
        nn.Linear(9, 9),
        nn.Tanh(),
        nn.Linear(9, 1)
    )

    def count_params(model):
        return sum(p.numel() for p in model.parameters())

    print("Однослойная параметры:", count_params(one_layer))
    print("Двухслойная параметры:", count_params(two_layer))

    def train(model, X, y, epochs=6000, lr=0.01):
        optimizer = optim.Adam(model.parameters(), lr=lr)
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=300,
            min_lr=1e-5
        )
        loss_fn = nn.MSELoss()

        for _ in range(epochs):
            optimizer.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            optimizer.step()
            scheduler.step(loss)

        return loss.item()

    loss_one = train(one_layer, X, y)
    loss_two = train(two_layer, X, y)

    print(f"Однослойная MSE = {loss_one:.8f}")
    print(f"Двухслойная MSE = {loss_two:.8f}")

    with torch.no_grad():
        y_pred_one = one_layer(X)
        y_pred_two = two_layer(X)

    plt.figure(figsize=(8,5))
    plt.plot(X.numpy(), y.numpy(), label='Target  $e^{-|x|}$', linewidth=2)
    plt.plot(X.numpy(), y_pred_one.numpy(), '--', label=f'One hidden layer ({i} neurons)')
    plt.plot(X.numpy(), y_pred_two.numpy(), '--', label='Two hidden layers')
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.title("Аппроксимация $f(x)=e^{-|x|}$")
    plt.grid(True)
    plt.show()